In [ ]:
%matplotlib inline
import os, sys
sys.path.insert(0, 'scripts')

import matplotlib.pyplot as plt
import pandas as pd
from sqlalchemy import create_engine

from analytics import (
    championship_trajectory, teammate_delta,
    qualifying_race_ols, pit_stop_efficiency, dnf_rate_model,
)
from charts import (
    championship as chart_championship, teammate_delta_chart,
    qualifying_regression, pit_stops_chart, reliability_chart,
)
from constants import TEAM_NAME, TEAM_COLORS
from load_data import _build_connection_string

try:
    from config import DB_CONFIG
except ImportError:
    DB_CONFIG = {'type': 'sqlite', 'filename': 'f1_analytics.db'}

engine = create_engine(_build_connection_string(DB_CONFIG))
print(f'Connected  ·  {TEAM_NAME}')

## Championship Progression
Cumulative driver points per round across all seasons.

In [ ]:
traj = championship_trajectory(engine)
chart_championship(traj, team_name=TEAM_NAME, colors=TEAM_COLORS)
plt.show()

## Teammate Head-to-Head
Mean finish-position delta in shared races (DNFs excluded). 95% t-interval; `*` p < 0.05, `**` p < 0.01, `***` p < 0.001, `ns` not significant.

In [ ]:
delta = teammate_delta(engine)
display(delta[['driver_a', 'driver_b', 'mean_delta', 'ci_lower', 'ci_upper', 'n', 'p_value']])
teammate_delta_chart(delta, colors=TEAM_COLORS)
plt.show()

## Qualifying → Race Regression
OLS regression of grid position on finish position. R² quantifies how strongly starting position determines outcome; slope gives expected positions gained or lost per grid row.

In [ ]:
ols_stats, scatter = qualifying_race_ols(engine)
p = ols_stats['p_value']
print(
    f"R\u00b2 = {ols_stats['r2']:.3f}   "
    f"slope = {ols_stats['slope']:.3f}   "
    f"p {'< 0.001' if p < 0.001 else f'= {p:.4f}'}   "
    f"n = {ols_stats['n']}"
)
qualifying_regression(scatter, ols_stats, colors=TEAM_COLORS)
plt.show()

## Pit Stop Efficiency
Per-driver stop times z-scored against the season field distribution. z < 0 = faster than the season average. Stops outside [15 s, 60 s] are excluded (safety-car pits, data errors).

In [ ]:
pit = pit_stop_efficiency(engine)
display(pit)
pit_stops_chart(pit, colors=TEAM_COLORS)
plt.show()

## DNF Rate Model
Poisson MLE for retirement rate per race with exact 95% confidence intervals (chi-squared exact method).

In [ ]:
dnf = dnf_rate_model(engine)
display(dnf[['driver', 'races', 'dnfs', 'rate', 'ci_lower', 'ci_upper']])
reliability_chart(dnf, colors=TEAM_COLORS)
plt.show()

## Export
Save all figures as 300 DPI PNG to `data/exports/charts/`.

In [ ]:
EXPORT_DIR = os.path.join('data', 'exports', 'charts')
os.makedirs(EXPORT_DIR, exist_ok=True)

def _save(fig, name):
    if fig is None:
        return
    path = os.path.join(EXPORT_DIR, f'{name}.png')
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved: {path}')

_save(chart_championship(championship_trajectory(engine), team_name=TEAM_NAME, colors=TEAM_COLORS), 'championship')
_save(teammate_delta_chart(teammate_delta(engine), colors=TEAM_COLORS), 'teammate_delta')
_ols, _scatter = qualifying_race_ols(engine)
_save(qualifying_regression(_scatter, _ols, colors=TEAM_COLORS), 'qualifying_ols')
_save(pit_stops_chart(pit_stop_efficiency(engine), colors=TEAM_COLORS), 'pit_stop_efficiency')
_save(reliability_chart(dnf_rate_model(engine), colors=TEAM_COLORS), 'reliability')